# Deepfake Detector — Colab Training
CLIP ViT-L/14 + DoRA + FFT fusion

**Runtime → Change runtime type → A100 GPU**

In [ ]:
# 1. Install dependencies
!pip install -q torch torchvision transformers peft albumentations opencv-python scikit-learn tqdm kagglehub huggingface_hub

In [ ]:
# 1b. Mount Google Drive and cd into your project folder
from google.colab import drive
drive.mount('/content/drive')

# Change this to match your folder path in Drive
%cd /content/drive/MyDrive/Colab Notebooks
!ls *.py

In [ ]:
# 2. Upload model.py and train.py
#    Drag both files into the file browser (folder icon on the left sidebar)
#    Then run this cell to verify they're there
import os
needed = ["model.py", "train.py"]
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print(f"Missing files: {missing}")
    print("Drag them into the Files panel on the left sidebar, then re-run this cell.")
else:
    print("All files found!")

In [ ]:
# 4. Download datasets
import kagglehub

print("Downloading Dataset 1: manjilkarki/deepfake-and-real-images...")
ds1_path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")
print(f"  → {ds1_path}")

print("Downloading Dataset 2: xhlulu/140k-real-and-fake-faces...")
ds2_path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print(f"  → {ds2_path}")

# Dataset 3: OpenRL/DeepFakeFace (diffusion) via HuggingFace
import os, zipfile, random
from pathlib import Path
from huggingface_hub import snapshot_download

ds3_path   = Path("~/.cache/huggingface/datasets/openrl_deepfakeface").expanduser()
train_real = ds3_path / "Train" / "Real"
train_fake = ds3_path / "Train" / "Fake"
val_real   = ds3_path / "Validation" / "Real"
val_fake   = ds3_path / "Validation" / "Fake"

_populated = all(
    d.exists() and any(d.iterdir())
    for d in (train_real, train_fake, val_real, val_fake)
)
if _populated:
    print(f"Dataset 3: already populated at {ds3_path}")
else:
    print("Downloading Dataset 3: OpenRL/DeepFakeFace (diffusion, ~5 GB)...")
    repo_path = Path(snapshot_download(repo_id="OpenRL/DeepFakeFace", repo_type="dataset"))
    extract_root = ds3_path / ".extracted"
    extract_root.mkdir(parents=True, exist_ok=True)
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    def _extract(zip_name):
        ed = extract_root / Path(zip_name).stem
        if ed.exists() and any(ed.iterdir()):
            return ed
        ed.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(repo_path / zip_name) as zf:
            zf.extractall(ed)
        return ed

    def _link(src, dst, prefix):
        dst.mkdir(parents=True, exist_ok=True)
        n = 0
        for p in src.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                rel = p.relative_to(src)
                name = f"{prefix}__" + "_".join(rel.parts)
                d = dst / name
                if not (d.exists() or d.is_symlink()):
                    os.symlink(p.resolve(), d)
                    n += 1
        return n

    for zn in ["wiki.zip"]:
        n = _link(_extract(zn), train_real, Path(zn).stem)
        print(f"  [real] {zn}: linked {n}")
    for zn in ["inpainting.zip", "insight.zip", "text2img.zip"]:
        n = _link(_extract(zn), train_fake, Path(zn).stem)
        print(f"  [fake] {zn}: linked {n}")

    rng = random.Random(42)
    for src, dst in [(train_real, val_real), (train_fake, val_fake)]:
        dst.mkdir(parents=True, exist_ok=True)
        files = sorted(p for p in src.iterdir() if p.is_file() or p.is_symlink())
        n_val = int(len(files) * 0.05)
        for p in rng.sample(files, n_val):
            p.rename(dst / p.name)
        print(f"  [val] moved {n_val} from {src.name}")

print(f"  → {ds3_path}")

In [ ]:
# 4. Patch dataset paths in train.py for Colab
import re

with open("train.py", "r") as f:
    code = f.read()

code = re.sub(
    r'DATASET_1\s*=\s*Path\(.*?\)',
    f'DATASET_1       = Path("{ds1_path}/Dataset")',
    code
)
code = re.sub(
    r'DATASET_2\s*=\s*Path\(.*?\)',
    f'DATASET_2       = Path("{ds2_path}/real_vs_fake/real-vs-fake")',
    code
)

with open("train.py", "w") as f:
    f.write(code)

print("Patched dataset paths in train.py")

In [ ]:
# 5. Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → A100 GPU")

In [ ]:
# 6. Train!
!python train.py

In [ ]:
# 7. Download trained model
from google.colab import files
import shutil
shutil.make_archive("model", "zip", "model")
files.download("model.zip")